# Wavelet subsample — uniform per session

Copy of `3.2_wavelet_subsample.ipynb` with the training-set construction changed. The
original is left untouched.

**What changed**

| | `3.2_wavelet_subsample` | this notebook |
|---|---|---|
| frames kept per session | 2,000 of 20,000, drawn with p ∝ KDE density of a per-session t-SNE | 2,000 **uniformly at random** from all valid frames |
| features | 32 columns (raw positions + 0.5–32 Hz) for the t-SNE; 3.3 then clusters on 20 | only the 20 columns 3.3 clusters on |
| normalisation | subsample z-scored by its **own** stats | nothing applied; the **full-session** stats (raw and log) are saved so 3.3 applies exactly what it later applies to every frame |

**Why not the density weighting.** Berman et al. 2014 (*J R Soc Interface*, "Mapping the
stereotyped behaviour of freely moving fruit flies") run t-SNE on 20,000 random points per
fly, watershed the density, and draw from each region **in proportion to the integral of the
density over that region**: stratified sampling that *preserves* each individual's
composition while making sure every region is represented. Drawing individual points with
p ∝ density instead weights them by density twice (they already occur in proportion to it).
On 25 sessions that made the training set's cluster make-up differ from the session's by a
total variation of 0.09, against 0.02 for a uniform draw, with the rarest vigorous cluster
at ×0.71. A uniform draw is Berman's scheme in expectation, without the t-SNE.

Every session still contributes the same number of frames, so a long session does not
outvote a short one.

In [ ]:
"""SHARED DEFINITIONS -- identical in 3.2_..._uniform and 3.3_..._uniform."""
import os
import numpy as np
import pandas as pd

DATA = '/home/ines/repositories/representation_learning_variability/paper-individuality/data/'

BANDS = ['0.5', '1.0', '2.0', '4.0', '8.0']       # the bands the clustering uses
FEATURES = {
    'paw':      [f'{p}_{a}{b}' for p in ('l_paw', 'r_paw') for a in 'xy' for b in BANDS],
    'left_paw': [f'l_paw_{a}{b}' for a in 'xy' for b in BANDS],
    'wheel':    [f'avg_wheel_vel{b}' for b in ('4.0', '8.0', '16.0', '32.0')],
}
WAVELETS = {       # VAR -> (folder, file prefix)
    'paw':      ('paw_wavelets/', 'paw_vel_wavelets_'),
    'left_paw': ('left_paw_wavelets/', 'paw_vel_wavelets_'),
    'wheel':    ('wheel_wavelets/', 'wheel_vel_wavelets_'),
}


def session_tags(var):
    """'<eid>_<mouse>' for every wavelet file of this VAR (top level only, as before)."""
    folder, prefix = WAVELETS[var]
    return sorted(f[len(prefix):] for f in os.listdir(DATA + folder) if f.startswith(prefix))


def load_session(var, tag):
    """-> (raw amplitudes of the frames with every feature finite, their Bin, the mask)."""
    folder, prefix = WAVELETS[var]
    df = pd.read_parquet(DATA + folder + prefix + tag, columns=FEATURES[var] + ['Bin'])
    X = df[FEATURES[var]].to_numpy(float)
    valid = np.isfinite(X).all(axis=1)
    return X[valid], df['Bin'].to_numpy()[valid], valid


In [ ]:
"""KNOBS"""
VAR   = 'paw'           # 'paw' | 'left_paw' | 'wheel'
N_SUB = 2000            # frames per session -- same as before, every session weighs the same
SEED  = 0
EPS   = 0.1             # log floor, in amplitude units: log(x + EPS). Paw amplitudes have a
                        # median of ~1-6 and a 1st percentile of ~0.01-0.07, so without a
                        # floor the log would stretch the near-zero tracking jitter of a still
                        # paw into the widest part of the space. 0.1 compresses that noise floor
                        # and leaves everything above ~0.3 essentially log-scaled.
                        # Chosen for PAW; revisit for wheel.

OUT = DATA + f'{VAR}_subsampled_wavelets_uniform/'
os.makedirs(OUT, exist_ok=True)
tags = session_tags(VAR)
print(f'{len(tags)} sessions -> {OUT}')

In [ ]:
"""SUBSAMPLE -- one .npz per session."""
rng = np.random.default_rng(SEED)
too_short = []
for tag in tags:
    X, bins, _ = load_session(VAR, tag)
    if len(X) < N_SUB:
        too_short.append((tag, len(X)))
        continue
    pick = np.sort(rng.choice(len(X), N_SUB, replace=False))
    L = np.log(X + EPS)
    np.savez(OUT + tag, sub=X[pick], bins=bins[pick], features=np.array(FEATURES[VAR]),
             n_valid=len(X), eps=EPS,
             mean_raw=X.mean(0), sd_raw=X.std(0),       # ddof=0, as scipy.stats.zscore
             mean_log=L.mean(0), sd_log=L.std(0))
print(f'saved {len(tags) - len(too_short)}; too short: {too_short}')